# Part 3 — Extra exercises if you want more fun

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from scipy.integrate import quad
from scipy.optimize import minimize_scalar, bisect
from scipy.stats import chi2, norm, truncnorm, uniform

from functions import generate_dataset, loglikelihood, get_lambdamu_distribution, get_Lambda_distribution

# Settings: font size for plot labels
matplotlib.rcParams['font.size'] = 14

In [ ]:
xmin, xmax, sigma = -10, 10, 1
background_pdf = uniform(loc=xmin, scale=xmax-xmin)

class sig_pdf:
    def __init__(self, mu, sigma):
        self.mu, self.sigma = mu, sigma

    def pdf(self, x, sigma=None):
        s = sigma if sigma is not None else self.sigma
        return truncnorm.pdf(x, xmin/s, xmax/s, loc=self.mu, scale=s)

    def rvs(self, size, sigma=None):
        s = sigma if sigma is not None else self.sigma
        return truncnorm.rvs(xmin/s, xmax/s, loc=self.mu, scale=s, size=size)
    
signal_pdf = sig_pdf(0, sigma)

## Asimov dataset approximation for the discovery potential

If you would like to quickly estimate discovery potentials without spending too much computational resources, you can use the following approximations:

- Approximate the $H_0$ distribution by a $\chi^2$ distribution with one degree of freedom (Wilks' theorem)
- Evaluate the median $H_1$ $\Lambda$ assuming a representative dataset (called __Asimov dataset__). In this case, the test statistic can be expressed as

$$\Lambda(\mu_s) \sim 2\int_{x_\mathrm{min}}^{x_\mathrm{max}} \left[\mu_s S(x) + \mu_b B(x)\right] \times \log\left(1 + \frac{\mu_s S(x)}{\mu_b B(x)}\right) - \mu_s S(x) \;\mathrm{d}x$$

In [ ]:
mu_background = 10

def asimov(mu_signal, mu_background):
    def asimov_integrand(x):
        s = mu_signal * signal_pdf.pdf(x)
        b = mu_background * background_pdf.pdf(x)
        return (s + b) * np.log(1 + s/b) - s
    res = quad(asimov_integrand, -10, 10)
    return 2 * res[0]

xmu = np.arange(0, 10, 0.01)
sqts_a = np.array([np.sqrt(asimov(mu, mu_background)) for mu in xmu])
plt.plot(xmu, sqts_a, label=r"$\sqrt{\mathrm{Asimov}\;\Lambda}$")
plt.xlim(0.1, 30)
plt.ylim(0.1, 30)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Signal expectation")
plt.ylabel("Significance")
plt.grid(which='both')
plt.legend()
plt.show()

**QUESTION:** Compare with the exact approach with pseudo-experiments.

## Parameter fitting


Sometimes, in addition to the expected number of signal events you might want to fit the shape of the PDF. Here we give an example where we fit both $\mu_s$ and the width of the Gaussian distribution.

**Question:** Compute the log-loglikelihood on a 2D grid (mus, sigmas) and extract the best-fit point.

In [ ]:
mu_signal = 10
mu_background = 20
sigma = 1.5

# Generate random dataset
data, nsig, nbackground = generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf)

if len(data)==0:
    raise RuntimeError("Zero events generated in this dataset. Try again...")

# Scan over numbers of signal events AND sigma
mus = np.arange(0, len(data), 0.1)
sigmas = np.arange(0.5, 5, 0.1)

loglike = None
mu_best = None
sigma_best = None

# Compute the likelihood as a function of mu_signal and sigma and find the best-fit point
# ...
loglike = ...

# Plot -log(likelihood)
plt.contourf(mus, sigmas, 2*abs(loglike-loglike.max()), levels=(0, 2.30, 6.18))
plt.scatter([mu_signal], [sigma], s=40, color='r', label="Injection")
plt.scatter([mu_best], [sigma_best], marker="*", s=40, color='yellow', label="Best fit")
plt.xlabel("Expected number of signal events")
plt.ylabel("$\\sigma$")
plt.legend(loc="best")

**Question**: Explain the choice of values for the contour.

## Systematic uncertainties


So far we have been missing one of the main troublemakers: systematic uncertainties. Let us assume that a parameter $\theta$ in the likelihood is affected by a systematic uncertainty $\sigma_\theta$. In a __frequentist__ framework, we account for this uncertainty by setting $\theta$ to the value that maximizes the likelihood:

$$\mathcal{L}(\mu, \hat\theta) = \mathrm{max}_\theta [\mathcal{L}(\mu, \theta) \times P(\theta)]$$

where $P(\theta)$ is a distribution normalized to $1$ that constrains that possible values $\theta$ can take (which I personally call a __hypocritical prior__). It is typically Gaussian, or sometimes uniform within a certain range.

With this method, the test-statistic becomes

$$\Lambda(\mu) = 2\log\frac{\mathcal{L}(\hat\mu, \hat\theta)}{\mathcal{L}\left(0, \hat{\hat\theta}\right)}.$$

Note that the likelihoods in the numerator and the denominator are maximized separately and hence will typically have different best-fit $\theta$ value (hence the hat piling).

**The hybrid Bayesian-Frequentist method**

For the frequentist likelihood method presented throughout this notebook, the frequentist treatment is the correct one and provides conservative performance estimates. However, this method can be very computationally intensive. A possible shortcut is to treat the systematic uncertainties in a Bayesian way. In this case, a prior probability distribution $P(\theta)$ is assumed for the $\theta$ parameter and the likelihood is marginalized over this prior:

$$\mathcal{L}(\mu) = \int \mathcal{L}(\mu, \theta) \times P(\theta)\; \mathrm{d}\theta.$$

This method is not recommended but can be used when systematic uncertainties are not expected to impact much the analysis.

**Example**: here we will consider a systematic uncertainty on $\mu_b$. $P(\mu_b)$ is assumed to be uniform over a given interval. 

Try varying the numbers of events and the width of the signal distribution. When do systematics matter?

In [ ]:
mu_signal = 3
mu_background = 5

# Range of sigma
bkg_syst = 0.5 # relative uncertainty
bkg_min, bkg_max = max(0, (1-3*bkg_syst) * mu_background), (1+3*bkg_syst) * mu_background

# Frequentist treatment: Treat background as a nuisance parameter
def loglike_nuisance(data, mu_signal):
    l1 = minimize_scalar(lambda x: -loglikelihood(data, mu_signal, x, signal_pdf, background_pdf) - norm.logpdf(x, mu_background, bkg_syst * mu_background), 
                         bounds=(bkg_min, bkg_max))
    return -l1.fun

# Bayesian treatment: Marginalize likelihood over prior
def loglike_bayes(data, mu_signal):
    l0 = loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf)
    integrand = lambda x: np.exp(loglikelihood(data, mu_signal, x, signal_pdf, background_pdf) - l0) * norm.pdf(x, mu_background, bkg_syst * mu_background)
    # Question: why shift by l0?
    l1 = quad(integrand, bkg_min, bkg_max)[0]
    return np.log(l1) + l0

# Generate random dataset
data, nsig, nbackground = generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf)

if len(data)==0:
    raise RuntimeError("Zero events generated in this dataset. Try again...")

# Scan over numbers of signal events
mus = np.arange(0, len(data), 0.1)

loglike_nosys = np.array([loglikelihood(data, mu, mu_background, signal_pdf, background_pdf) for mu in mus])
loglike_sys_freq = np.array([loglike_nuisance(data, mu) for mu in mus])
loglike_sys_bayes = np.array([loglike_bayes(data, mu) for mu in mus])

loglike_nosys_max = loglike_nosys.max()
loglike_sys_freq_max = loglike_sys_freq.max()
loglike_sys_bayes_max = loglike_sys_bayes.max()

# Plot lambda(mu)
plt.plot(mus, 2*abs(loglike_nosys - loglike_nosys.max()), label="No systematics")
plt.plot(mus, 2*abs(loglike_sys_freq - loglike_sys_freq.max()), label="Frequentist systematics")
plt.plot(mus, 2*abs(loglike_sys_bayes - loglike_sys_bayes.max()), color='tab:red', linestyle='--', label="Bayesian systematics")
plt.axvline(nsig, color='r')

plt.ylim(0, 1)
plt.xlim(0, len(data))
plt.grid(which='both')

plt.xlabel("Expected number of signal events")
plt.ylabel(r"$-\log\dfrac{\mathcal{L}}{\mathcal{L}_\mathrm{max}}$")
plt.legend()

## Compute confidence intervals and sensitivity with different approaches

Confidence intervals and sensitivities are obtained in the same way, using some criteria to know which part of the parameter space is allowed / rejected.
* For confidence interval, we use the observation as a point of reference (to define the threshold).
* For sensitivity, we consider background-only pseudo-experiments and take the median out of it.

The computations below are detailed for the sensitivity at 90% CL, but it can easily be adapted for derivation of confidence intervals or upper limits.

#### Cowan et al

In [ ]:
mu_background = 10
Npseudo_experiments_H0 = 1000
Npseudo_experiments_H1 = 5000

fig, ax = plt.subplots(1, 4, figsize=(12, 3), sharey=True, gridspec_kw={"wspace":0.04})

for i, mu_signal in enumerate([2, 5, 10, 15]):
    lambdamu_H1 = get_lambdamu_distribution(mu_signal, mu_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H0)
    lambdamu_H0 = get_lambdamu_distribution(mu_signal, 0, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H1)
    bins = np.linspace(-0.01, lambdamu_H0.max(), 51)
    x = np.linspace(bins[0], bins[-1], 1001)
    ax[i].plot(x, chi2.pdf(x, 1), color="cyan")
    ax[i].hist(lambdamu_H1, bins, density=True, histtype="step", color="blue")
    ax[i].hist(lambdamu_H0, bins, density=True, histtype='step', color='black')
    ax[i].axvline(np.median(lambdamu_H0), linestyle="dotted", lw=2, color='black')
    ax[i].axvline(np.percentile(lambdamu_H1, 90), linestyle="dashed", color='blue')
    ax[i].set_xlabel(r"$\lambda_{\mu}$")
    ax[i].set_title(rf"$\mu = {mu_signal}$")

ax[0].set_ylabel("Number of p.-e.s")
ax[0].set_yscale('log')
ax[0].set_ylim(0.5/Npseudo_experiments_H1, None)
handles = [
    matplotlib.patches.Patch(facecolor="blue", label=r"$H_1(\mu)$"),
    matplotlib.patches.Patch(facecolor="black", label=r"$H_0$"),
    matplotlib.lines.Line2D([], [], color="blue", ls="dashed", label="90% threshold"),
    matplotlib.lines.Line2D([], [], color="black", ls="dotted", label="Median"),
    matplotlib.lines.Line2D([], [], color="cyan", label=r"$\chi^2$")
]
fig.legend(handles=handles, bbox_to_anchor=(0.9, 0.5), loc="center left")

In [ ]:
def find_sensitivity(mu, mu_background):
    lambdamu_H1 = get_lambdamu_distribution(mu, mu, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H0)
    lambdamu_H0 = get_lambdamu_distribution(mu, 0, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H1)
    return np.percentile(lambdamu_H1, 90) - np.median(lambdamu_H0)

precision = 2
print("Sensitivity(Cowan) =", round(bisect(find_sensitivity, 0, mu_background, args=(mu_background,), xtol=10**-precision), precision))

#### Cowan + Wilks'

In [ ]:
def find_sensitivity_wilks(mu, mu_background):
    thr_wilks = chi2.ppf(0.90, df=1)
    lambdamu_H0 = get_lambdamu_distribution(mu, 0, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H1)
    return thr_wilks - np.median(lambdamu_H0)

precision = 2
print("Sensitivity(Cowan + Wilks) =", round(bisect(find_sensitivity_wilks, 0, mu_background, args=(mu_background,), xtol=10**-precision), precision))

#### Sensitivity using $\Lambda = \lambda_0$

In [ ]:
Lambda_H0 = get_Lambda_distribution(0, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H0)
Lambdas_H1 = []
mus_signal = [2, 5, 10, 15]
for mu_signal in mus_signal:
    Lambdas_H1.append(get_Lambda_distribution(mu_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H1))

fig, ax = plt.subplots(1, 4, figsize=(12, 3), sharey=True, gridspec_kw={"wspace":0.04})

bins = np.linspace(-0.01, np.max(Lambdas_H1), 51)
for i, mu_signal in enumerate(mus_signal):
    ax[i].hist(Lambda_H0, bins, density=True, histtype='step', color='black')
    ax[i].hist(Lambdas_H1[i], bins, density=True, histtype="step", color="blue")
    ax[i].axvline(np.median(Lambda_H0), linestyle="dotted", lw=2, color='black')
    ax[i].axvline(np.percentile(Lambdas_H1[i], 10), linestyle="dashed", color='blue')
    ax[i].set_xlabel(r"$\Lambda$")
    #ax[i].set_title(rf"$\mu_{{\text{{signal}}}} = {mu_signal}$")

ax[0].set_ylabel("Number of p.-e.s")
ax[0].set_yscale('log')
ax[0].set_ylim(0.5/Npseudo_experiments_H1, None)
handles = [
    matplotlib.patches.Patch(facecolor="blue", label=r"$H_1(\mu)$"),
    matplotlib.patches.Patch(facecolor="black", label=r"$H_0$"),
    matplotlib.lines.Line2D([], [], color="blue", ls="dashed", label="10% threshold"),
    matplotlib.lines.Line2D([], [], color="black", ls="dotted", label="Median"),
]
fig.legend(handles=handles, bbox_to_anchor=(0.9, 0.5), loc="center left")

In [ ]:
Lambda_H0 = get_Lambda_distribution(0, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H0)

def find_sensitivity_Lambda(mu, mu_background):
    Lambda_H1 = get_Lambda_distribution(mu, mu_background, signal_pdf, background_pdf, Npseudo_experiments_H1)
    return np.percentile(Lambda_H1, 10) - np.median(Lambda_H0)

precision = 2
print("Sensitivity(Lambda) =", round(bisect(find_sensitivity_Lambda, 0, mu_background, args=(mu_background,), xtol=10**-precision), precision))